# Data Wrangling: Join, Combine, and Reshape

In many applications, data may be spread across a number of files or databases, or be arranged in a form that is not convenient to analyze. This chapter focuses on tools ot help combine, join, and rearrange data. 

# Hierarchical Indexing 

*Hierarchical indexing* is an import feature of pandas that enables you to have multiple (two or more) index *levels* on an axis. Another way of thinking about it is that it provides a way for you to work with higher dimensional data in a lower dimensional form. Let's start with a simple example:

Create a Series with a list of lists (or arrays) as the index:

In [2]:
import pandas as pd 
import numpy as np 

In [10]:
data = pd.Series(np.random.uniform(size=9),
                index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])

print(data)

a  1    0.816714
   2    0.599755
   3    0.481131
b  1    0.403910
   3    0.072647
c  1    0.810813
   2    0.723148
d  2    0.354547
   3    0.226610
dtype: float64


What you're seeing is a prettified view of a Series with a `` MultiIndex`` as its index. The "gaps" in the index display mean "use the label directly above":

In [11]:
data.index

MultiIndex([('a', 1),
            ('a', 2),
            ('a', 3),
            ('b', 1),
            ('b', 3),
            ('c', 1),
            ('c', 2),
            ('d', 2),
            ('d', 3)],
           )

With a hierarchially indexed object, so-called *partial* indexing is possible, enabling you to concisely select subsets of the data:

In [12]:
data["b"]

1    0.403910
3    0.072647
dtype: float64

In [13]:
data["b":"c"]

b  1    0.403910
   3    0.072647
c  1    0.810813
   2    0.723148
dtype: float64

In [14]:
data.loc[["b", "d"]]

b  1    0.403910
   3    0.072647
d  2    0.354547
   3    0.226610
dtype: float64

Selection is even possible from an "inner" level. Here I select all of the values having the value 2 from the second index level:

In [15]:
data.loc[:, 2]

a    0.599755
c    0.723148
d    0.354547
dtype: float64

Hierarchial indexing plays an important role in reshaping data and in group-based operations like forming a pivot table. For example, you can rearrange this data into a DataFrame using its ``unstack`` method:

In [16]:
data.unstack()

,1,2,3
a,0.816714,0.599755,0.481131
b,0.403910,NaN,0.072647
c,0.810813,0.723148,NaN
d,NaN,0.354547,0.226610


The inverse operation of ``unstack`` is ``stack``:

In [17]:
data.unstack().stack()

a  1    0.816714
   2    0.599755
   3    0.481131
b  1    0.403910
   3    0.072647
c  1    0.810813
   2    0.723148
d  2    0.354547
   3    0.226610
dtype: float64

``stack`` and ``unstack`` with be explored in more detail later. 

With a DataFrame, either axis can have a hierarchial index:

In [20]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                    index =[["a", "a", "b", "b"], [1, 2, 1, 2]],
                    columns=[["Ohio", "Ohio", "Colorado"],
                            ["Green", "Red", "Green"]])

frame

Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11

In [21]:
print(frame)

     Ohio     Colorado
    Green Red    Green
a 1     0   1        2
  2     3   4        5
b 1     6   7        8
  2     9  10       11


The hierarchial levels can have names (as strings or any Python objects). If so, these will show up in the console output:

In [23]:
frame.index.names = ["key1", "key2"]

frame.columns.names = ["state", "color"]

frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

These names supersede the ``name`` attribute, which is used only with single-level indexes. 

You can see how many levels an index has by accessing its ``nlevels`` attribute:

In [24]:
frame.index.nlevels

2

With partial column indexing, you can similarly select groups of columns:

In [25]:
frame["Ohio"]

color      Green  Red
key1 key2            
a    1         0    1
     2         3    4
b    1         6    7
     2         9   10

A ``MultiIndex`` can be created by itself and then reused; the columns in the preceding DataFrame with level names could also be created like this:

In [26]:
pd.MultiIndex.from_arrays([["Ohio", "Ohio", "Colorado"],
                        ["Green", "Red", "Green"]], 
                        names = ["state", "color"])

MultiIndex([(    'Ohio', 'Green'),
            (    'Ohio',   'Red'),
            ('Colorado', 'Green')],
           names=['state', 'color'])

## Reordering and Sorting Levels 

At times you may need to rearrange the order of the levels on an axis or sort the data by the values in one specific level. The ``swaplevel`` method takes two level numbers or names and returns a new object with the levels interchanged (but the data is otherwise unaltered):

In [27]:
frame.swaplevel("key1", "key2")

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

``sort_index`` by default sorts the data lexicographically using all the index levels, but you can choose to use only a single level or a subset of levels to sort by passing the ``level`` argument. For example:

In [28]:
frame.sort_index(level=1)

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [29]:
frame.swaplevel(0, 1).sort_index(level=0)

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

### Summary Statistics by Level

Many descriptive and summary statistics on DataFrame and Series have a ``level`` option in which you can specify the level you want to aggregate by on a particular axis. Consider the above DataFrame; we can go aggregate by level on either the rows or columns, like so:

In [30]:
frame.groupby(level="key2").sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

In [34]:
frame.T.groupby(level="color").sum()

key1   a      b    
key2   1  2   1   2
color              
Green  2  8  14  20
Red    1  4   7  10

### Indexing with a DataFrame's columns

It's not unusual to want to use one or more columns from a DataFrame as the row index; alternatively, you may wish to move the row index into the DataFrame's columns. Here's an example DataFrame:

In [37]:
frame = pd.DataFrame({"a": range(7), "b": range(7, 0, -1),
                    "c": ["one", "one", "one", "two", 'two', "two", "two"],
                    "d": [0, 1, 2, 0, 1, 2, 3]})

frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


DataFrame's ``set_index`` function will create a new DataFrame using one or more of its columns as the index:

In [38]:
frame2 = frame.set_index(["c", "d"])

frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1

By default, the columns are removed from the DataFrame, though you can leave them in by passing ``drop = False`` to ``set_index``:

In [39]:
frame.set_index(["c", "d"], drop=False)


a  b    c  d
c   d              
one 0  0  7  one  0
    1  1  6  one  1
    2  2  5  one  2
two 0  3  4  two  0
    1  4  3  two  1
    2  5  2  two  2
    3  6  1  two  3

``reset_index``, on the other hand, does the opposite of ``set_index``; the hierarchial index levels are moved into the columns:

In [41]:
frame2.reset_index()

,c,d,a,b
0,one,0,0,7
1,one,1,1,6
2,one,2,2,5
3,two,0,3,4
4,two,1,4,3
5,two,2,5,2
6,two,3,6,1


# Combining and Merging Datasets